In [1]:
import pandas as pd

train_data_encoded = pd.read_csv('../data/train_encoded.csv')
test_data_encoded = pd.read_csv('../data/test_encoded.csv')
# train_data_encoded.info()

# since random forest is resistent to outliers, we just skip these steps

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from tqdm.auto import tqdm
import hashlib
import pickle

# 1. Load & prepare data
X = train_data_encoded.drop('y', axis=1).values
y = train_data_encoded['y']

# 2. CV and parameter grid setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'n_estimators':     [200, 500, 700],
    'min_samples_leaf': [1, 5, 10],
    'max_depth':        [None, 10, 20],
    'max_features':     ['sqrt', 'log2', 0.3]
}
grid = list(ParameterGrid(param_grid))

# 3. Create a cache dictionary
cache = {}

# Optional: Try loading existing cache from file
try:
    with open('rf_grid_cache.pkl', 'rb') as f:
        cache = pickle.load(f)
    print("Cache loaded from file.")
except FileNotFoundError:
    print("No cache file found, starting fresh.")

# 4. Manual grid search with caching
results = []

for params in tqdm(grid, desc="Hyperparameter tuning"):
    # Generate a unique key for the parameter combination
    params_str = str(sorted(params.items()))
    # print(params_str)
    params_hash = hashlib.md5(params_str.encode()).hexdigest()
    
    if params_hash in cache:
        avg_mse = cache[params_hash]
        print(f"Found cached result for {params} -> MSE: {avg_mse:.4f}")
    else:
        fold_mse = []
        for train_idx, val_idx in cv.split(X, y):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]

            rf = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            rf.fit(X_tr, y_tr)
            y_val_pred = rf.predict(X_val)

            mse = mean_squared_error(y_val, y_val_pred)
            fold_mse.append(mse)

        avg_mse = np.mean(fold_mse)
        cache[params_hash] = avg_mse  # Save new result
        print(f"Computed and cached result for {params} -> MSE: {avg_mse:.4f}")

    results.append({**params, 'avg_mse': avg_mse})

# 5. Save updated cache
with open('rf_grid_cache.pkl', 'wb') as f:
    pickle.dump(cache, f)

# 6. Summarize results
results_df = pd.DataFrame(results).sort_values('avg_mse').reset_index(drop=True)
print("\nTop 5 hyperparameter settings by CV-estimated MSE:")
print(results_df.head(5).to_string(index=False))

best_params = results_df.loc[0, list(param_grid.keys())].to_dict()

# Fix NaNs and floats if needed
best_params = {
    k: (None if pd.isna(v) else int(v) if k in ['n_estimators', 'min_samples_leaf', 'max_depth'] else v)
    for k, v in best_params.items()
}

print(f"\nBest params: {best_params}")
print(f"Best CV-estimated MSE: {results_df.loc[0,'avg_mse']:.4f}")


Cache loaded from file.


Hyperparameter tuning: 100%|██████████| 81/81 [00:00<00:00, 77993.26it/s]

Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200} -> MSE: 4852.1300
Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 500} -> MSE: 4851.4222
Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 700} -> MSE: 4867.0741
Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 200} -> MSE: 4993.4880
Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 500} -> MSE: 4992.2758
Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 700} -> MSE: 4990.6575
Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 200} -> MSE: 5174.5913
Found cached result for {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1